In [ ]:
import cv2
import os
import numpy as np
import pickle
from cv2 import face

# capturar 
def capturar_rostros(nombre_persona, total_fotos=20):
    ruta = f"dataset/{nombre_persona}"
    os.makedirs(ruta, exist_ok=True)
    cap = cv2.VideoCapture(0)
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    count = 0
    print(f"Capturando {total_fotos} rostros para {nombre_persona}...")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        rostros = face_cascade.detectMultiScale(gray, 1.3, 5)
        for (x, y, w, h) in rostros:
            rostro = gray[y:y+h, x:x+w]
            cv2.imwrite(f"{ruta}/{count}.jpg", rostro)
            count += 1
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, f"Capturas: {count}/{total_fotos}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow("Captura de rostros - ESC para salir", frame)
    
        if count >= total_fotos or cv2.waitKey(1) == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)  

def entrenar_modelo(dataset_path="dataset"):
    imagenes, etiquetas, mapa_etiquetas = [], [], {}
    label_id = 0
    for persona in os.listdir(dataset_path):
        ruta_persona = os.path.join(dataset_path, persona)
        if not os.path.isdir(ruta_persona):
            continue
        mapa_etiquetas[label_id] = persona
        for nombre_imagen in os.listdir(ruta_persona):
            img = cv2.imread(os.path.join(ruta_persona, nombre_imagen), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                imagenes.append(img)
                etiquetas.append(label_id)
        label_id += 1
    reconocedor = face.LBPHFaceRecognizer_create()
    reconocedor.train(imagenes, np.array(etiquetas))
    os.makedirs("recognizer", exist_ok=True)
    reconocedor.save("recognizer/trainer.yml")
    with open("recognizer/labels.pickle", "wb") as f:
        pickle.dump(mapa_etiquetas, f)



def reconocer_rostros(umbral_confianza=70):
    reconocedor = face.LBPHFaceRecognizer_create()
    reconocedor.read("recognizer/trainer.yml")
    with open("recognizer/labels.pickle", "rb") as f:
        mapa_etiquetas = pickle.load(f)
    cap = cv2.VideoCapture(0)
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    print("Iniciando reconocimiento. Presiona ESC para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        rostros = face_cascade.detectMultiScale(gray, 1.3, 5)
        for (x, y, w, h) in rostros:
            rostro = gray[y:y+h, x:x+w]
            id_, confianza = reconocedor.predict(rostro)
            nombre = mapa_etiquetas[id_] if confianza < umbral_confianza else "Desconocido"
            color = (0, 255, 0) if nombre != "Desconocido" else (0, 0, 255)
            cv2.putText(frame, nombre, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.imshow("Reconocimiento de rostros - ESC para salir", frame)
        if cv2.waitKey(1) == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

if __name__ == "__main__":
    capturar_rostros("mateo", total_fotos=20)
    entrenar_modelo()
    reconocer_rostros()